# 05 — Reasoning Effort

**Goal:** let the model **think harder** on hard problems, with one simple switch.

Modern LLMs can spend extra tokens "thinking" before they answer. The problem is that every provider exposes this differently:

- **OpenAI** takes a word: `low`, `medium`, or `high`.
- **Anthropic** takes a **token budget** (how many tokens to spend thinking).
- **Google** takes a thinking config object.
- **OpenRouter** takes a reasoning object.

orqest collapses all four into **one knob**: `reasoning="high"`. You set it once, and orqest translates it to whatever the model needs.

In this notebook you will:

1. See the translation in action — one input, four provider shapes.
2. Watch the setting flow into the agent's actual model config.
3. Run an honest A/B test on a hard problem at `low` vs `high` effort.
4. Save tokens by setting reasoning **per step** in a pipeline: cheap steps get no thinking, hard steps get a lot.

**Before you start:** you need a `.env` at the repo root with `LLM_API_KEY` and `LLM_MODEL`.


## 1. Load the config

In [1]:
from orqest import load_config

config = load_config()
print(f"Model: {config.llm_model}")

Model: openai:gpt-5.2


## 2. One knob, four shapes

The function `resolve_reasoning_settings` is pure translation. You give it a provider name and an effort level. It hands back the settings dict that provider expects. No LLM is called.

Run the cell below and look at the four very different outputs.


In [2]:
from orqest.utils.reasoning import resolve_reasoning_settings

for provider in ["openai", "anthropic", "google", "openrouter"]:
    print(f"{provider:11s} -> {resolve_reasoning_settings(provider, 'high')}")

openai      -> {'openai_reasoning_effort': 'high'}
anthropic   -> {'anthropic_thinking': {'type': 'enabled', 'budget_tokens': 24576}, 'max_tokens': 32768}
google      -> {'google_thinking_config': {'thinking_budget': 24576, 'include_thoughts': True}, 'max_tokens': 32768}
openrouter  -> {'openrouter_reasoning': {'effort': 'high', 'enabled': True}}


One vocabulary in (`"high"`), four shapes out:

- **openai** gets the categorical setting `openai_reasoning_effort` — pass-through.
- **anthropic** and **google** get a **token budget** derived from the effort level. They also get a `max_tokens` value: Anthropic *requires* `max_tokens > budget_tokens`, otherwise the call fails. orqest sets it for you so reasoning works out of the box.
- **openrouter** gets its own `openrouter_reasoning` object.

Once you write `reasoning="high"`, you never deal with these shapes again. The provider divergence stops here.


## 3. The knob takes effect

`reasoning` is just a constructor argument on `BaseAgent`. orqest translates it once (using your model's provider), and merges it into the agent's `model_settings`. If you also set explicit settings of your own, **your explicit keys win** on conflict.

You don't have to call the LLM to verify any of this. Just inspect `agent.agent.model_settings` after construction.

We will build three agents to make the merge behaviour visible:

1. **plain** — no reasoning setting at all.
2. **thinking** — `reasoning="high"`.
3. **mixed** — `reasoning="high"` plus a manual `temperature=0.2`.


First, the agent class and a small builder helper.

In [ ]:
from pydantic import BaseModel, Field
from pydantic_ai.settings import ModelSettings
from orqest.agents import BaseAgent, GlobalState


class Solution(BaseModel):
    """A worked answer: the method, the steps, the final number."""
    approach: str = Field(description="One sentence: the method used.")
    steps: list[str] = Field(description="The working, one concise step per item.")
    final_answer: int = Field(description="The final numeric answer.")


class SolverAgent(BaseAgent[GlobalState, Solution]):
    async def _run_implementation(self, state: GlobalState, **kwargs) -> Solution:
        result = await self.call_model(state.get_latest_message("user"), state)
        return result.output


def build_solver(reasoning=None, model_settings=None) -> SolverAgent:
    """A SolverAgent on the configured model, with an optional reasoning effort."""
    return SolverAgent(
        agent_name="solver",
        system_prompt="You are a careful mathematician. Show each step.",
        output_type=Solution,
        model=config.llm_model,
        api_key=config.llm_api_key,
        reasoning=reasoning,
        model_settings=model_settings,
    )


print("Solver class and builder ready.")

Now build the three variants and inspect the model settings each one ended up with.

In [3]:
plain    = build_solver()
thinking = build_solver(reasoning="high")
mixed    = build_solver(reasoning="high", model_settings=ModelSettings(temperature=0.2))

print("reasoning=None        ->", plain.agent.model_settings)
print("reasoning='high'      ->", thinking.agent.model_settings)
print("'high' + temperature  ->", mixed.agent.model_settings)

reasoning=None        -> None
reasoning='high'      -> {'openai_reasoning_effort': 'high'}
'high' + temperature  -> {'openai_reasoning_effort': 'high', 'temperature': 0.2}


Three observations:

- `reasoning=None` leaves the provider defaults alone.
- `reasoning="high"` adds the provider-specific key.
- The mixed case shows the merge: both the reasoning key **and** the explicit `temperature` are present.

Our model is OpenAI, so we see `openai_reasoning_effort`. If we switched to `anthropic:...`, the **same** agent code would produce `anthropic_thinking` instead. The translation happens automatically.


## 4. An honest A/B test

Now we actually run the model. We pick a **hard combinatorial** problem so that thinking can plausibly help:

> "How many distinct ways can you make exactly one US dollar using pennies, nickels, dimes, and quarters?"

The correct answer is **242**. We will run the same agent twice — once at `low` effort, once at `high` — and measure both **correctness** and **latency**.


In [4]:
import time

PROBLEM = (
    "How many distinct ways can you make exactly one US dollar (100 cents) "
    "using any number of pennies (1c), nickels (5c), dimes (10c), and "
    "quarters (25c)? Order does not matter."
)


async def solve(effort):
    agent = build_solver(reasoning=effort)
    state = GlobalState()
    state.add_message("user", PROBLEM)

    started = time.perf_counter()
    out = await agent.run(state)
    elapsed = time.perf_counter() - started

    mark = "correct" if out.final_answer == 242 else "WRONG"
    print(f"[{effort:>4}]  answer={out.final_answer} ({mark})  {elapsed:4.1f}s")
    print(f"         approach: {out.approach}")
    return out, elapsed


low_out,  low_t  = await solve("low")
high_out, high_t = await solve("high")

[ low]  answer=242 (correct)  12.6s
         approach: Reduce by 5 cents and sum over allowable numbers of quarters and dimes; pennies are then forced by the remainder.


[high]  answer=242 (correct)  14.1s
         approach: Casework on number of quarters; for each case count (dimes,nickels) pairs since pennies are forced.


### The honest read

Both efforts land on 242. The model used here (`gpt-5.2`) is already strong enough that its **floor** clears this problem.

What `high` reliably buys is more deliberation, which you pay for in **latency** (and tokens). It is **not** a guaranteed correctness boost on every problem.

So `reasoning` is a **dial, not a guarantee**. Spend it where the problem sits at the edge of the model's ability, and **measure** — don't assume "higher is always better". (This is the same dogfooding honesty as notebook 01: the feature does what it says, but you still own the calibration.)


## 5. Spend the budget where it matters

The `reasoning` setting lives on the **agent**. So in a multi-step workflow, you can give cheap steps no budget and the hard step a high one. Same workflow, far less spend.

Here is a `Pipeline` that does exactly that:

1. A **triage** agent — `reasoning=None`. It just restates the user's problem cleanly. No thinking needed.
2. A **solver** agent — `reasoning="high"`. This is the hard step.

A tiny adapter (`to_prompt`) bridges the triage output into the solver's prompt.


First, build the triage agent.

In [ ]:
from orqest.orchestration import Pipeline


class Restatement(BaseModel):
    """A problem, cleaned up and made explicit."""
    restated: str = Field(description="The problem restated precisely, with every quantity named.")


class TriageAgent(BaseAgent[GlobalState, Restatement]):
    async def _run_implementation(self, state: GlobalState, **kwargs) -> Restatement:
        result = await self.call_model(state.get_latest_message("user"), state)
        return result.output


# Cheap step: no reasoning budget.
triage = TriageAgent(
    agent_name="triage",
    system_prompt="Restate the user's problem precisely. Name every quantity. Do not solve it.",
    output_type=Restatement,
    model=config.llm_model,
    api_key=config.llm_api_key,
)

# Expensive step: high reasoning budget.
solver = build_solver(reasoning="high")

print("Triage and solver ready.")

Now assemble the pipeline and run a word problem through it. The triage step thinks lightly; the solver step thinks hard.

In [5]:
async def to_prompt(r: Restatement) -> str:
    """Adapter: bridge the triage output into the solver's prompt."""
    return f"Solve this problem, showing each step:\n\n{r.restated}"


pipeline = Pipeline([triage, to_prompt, solver], name="triage_then_solve")

WORD_PROBLEM = (
    "A bookstore sells novels for $12 and comics for $5. On Monday it sold "
    "17 items in total for $148. How many novels did it sell?"
)

result = await pipeline.run(WORD_PROBLEM)
print(f"approach: {result.approach}")
for i, step in enumerate(result.steps, 1):
    print(f"  {i}. {step}")
print(f"final_answer (novels): {result.final_answer}")

approach: Solve the system by substitution.
  1. From n + c = 17, solve for c: c = 17 − n.
  2. Substitute into 12n + 5c = 148: 12n + 5(17 − n) = 148.
  3. Distribute: 12n + 85 − 5n = 148.
  4. Combine like terms: 7n + 85 = 148.
  5. Subtract 85 from both sides: 7n = 63.
  6. Divide by 7: n = 9.
final_answer (novels): 9


The triage step used the provider defaults — no extra budget. The solver step used `reasoning="high"`.

Because `reasoning` lives on the agent itself, **the orchestration primitives never need to know about it**. `Pipeline`, `Parallel`, `Router` — none of them touch this setting. You configure it once, where the work is hard.


## Recap

Here is the picture:

1. `reasoning` is a single switch on every `BaseAgent`. Levels: `minimal`, `low`, `medium`, `high`.
2. orqest translates that single value into whatever the underlying provider expects.
3. If you also set explicit `model_settings`, your keys win on conflict.
4. The setting lives on the agent, so you can mix cheap and expensive steps in the same workflow.

**One sharp edge to know.** orqest translates; it does **not** validate that the model can actually reason. `gpt-5.2` accepts `low`/`medium`/`high` but **rejects `minimal`** with a 400 error. Each model supports a slightly different effort vocabulary. Pair `reasoning` with a model you have checked.

**One forward-looking note.** When pydantic-ai ships its unified `Thinking` capability, orqest's `ReasoningEffort` will map onto it 1:1, and the translation layer can be removed entirely. Your agent code (`reasoning="high"`) will not change.
